#### Workflow Objectives: 
The primary focus of this phase is to determe the transcription factor (TF) activity and linking accessibility to gene expression.

* **Motif & Deviation Analysis**: Scan for TF motifs from the JASPAR2020 database using TFBSTools and motifmatchr. Calculate accessibility deviations across conditions with chromVAR (v1.26.0).
* **Differential Accessibility**: Identify statistically significant differences in accessibility between samples using DESeq2 (v1.44.0).
* **Footprinting**: Identify TF "footprints" (narrow regions of protection within peaks) to infer active binding using MotifDb.
* **Multi-Omics Integration**: Correlate ATAC-seq peaks with RNA-seq data to validate functional regulatory links.

#### Data Processing & Implementation
These advanced analytical steps are performed within the R/Bioconductor environment, integrating standardized genomic objects (such as GRanges and SummarizedExperiment) to ensure statistical rigor and reproducibility across your ATAC-seq workflow.

#### Step 1: Summarizing read counts from BAM files
Quantify the accessibility of each genomic region by counting the number of filtered, non-redundant fragments overlapping the master peak set.

In [ ]:
# Load libraries
library(GenomicAlignments); library(TxDb.Hsapiens.UCSC.hg38.knownGene); library(biomaRt)
library(DESeq2)

# Identify and import all MACS2 narrowPeak files recursively
peaks <- list.files(path = "macs2", pattern = "peaks.narrowPeak", recursive = T, full.names = T)

# Convert raw peak files into GRanges objects for genomic manipulation
# Simple = TRUE extracts only the core coordinates (Chr, Start, End)
peaks <- lapply(peaks, ChIPQC:::GetGRanges, simple = TRUE)

# Create a non-redundant (union) peak set
# unlist/GRangesList combines all peaks; reduce() merges overlapping intervals into a single representative peak to prevent double-counting.
peaks_nR <- GenomicRanges::reduce(unlist(GRangesList(peaks)))

# Determine which individual samples contribute to each consensus peak
peaks_overlap <- list()
for (i in 1:length(peaks)) {
  peaks_overlap[[i]] <- peaks_nR %over% peaks[[i]]
}

# Build an overlap matrix to track peak presence across groups
peaks_overlap_matrix <- do.call(cbind, peaks_overlap)
colnames(peaks_overlap_matrix) <- bamfile.label
mcols(peaks_nR) <- peaks_overlap_matrix

# Check total number of non-redundant peaks before filtering
length(peaks_nR)

# Remove artifacts and mitochondrial noise
# Exclude ENCODE Blacklist regions and ChrM to ensure high-quality regulatory targets
nrToCount <- peaks_nR[!peaks_nR %over% blkList & !seqnames(peaks_nR) %in% "chrM"]

# Check final number of peaks ready for count summarization
length(nrToCount)
111310 

 111,310 represents the total number of unique, high-quality genomic regions (peaks) that survived your filtering process and are now ready for differential analysis


In [ ]:
# Organize BAM files and create a pointer list
# yieldSize = 5e+06 reads chunks at a time to manage memory during counting
bams <- c(paste("bowtie2_results/filtered_bam/Sample", 1:4, "_quality_controlled.bam", sep = ""),
          paste("bowtie2_results/filtered_bam/Sample", 5:8, "_quality_controlled.bam", sep = ""),
          paste("bowtie2_results/filtered_bam/Sample", 9:12, "_quality_controlled.bam", sep = ""),
          paste("bowtie2_results/filtered_bam/Sample", 13:16, "_quality_controlled.bam", sep = ""),
          paste("bowtie2_results/filtered_bam/Sample", 17:20, "_quality_controlled.bam", sep = ""))

bamFl <- BamFileList(bams, yieldSize = 5e+06)

# Quantify reads in the non-redundant (consensus) peak set
# singleEnd = FALSE ensures the tool treats your paired-end data correctly
counts <- summarizeOverlaps(features = peaks_nR,
                            reads = bamFl,
                            ignore.strand = FALSE,
                            singleEnd = FALSE)

# Label the count matrix by experimental replicate
colnames(counts) <- c(paste("Group1_Rep", 1:4, sep = ""), 
                      paste("Group2_Rep", 1:4, sep = ""),
                      paste("Group3_Rep", 1:4, sep = ""),
                      paste("Group4_Rep", 1:4, sep = ""),
                      paste("Group5_Rep", 1:4, sep = ""))

# Prepare metadata and build the DESeq2 object
Group <- factor(c(rep("Group1", 4), rep("Group2", 4), rep("Group3", 4), rep("Group4", 4), rep("Group5", 4)))
metaData <- data.frame(Group, row.names = colnames(counts))

atacDDS <- DESeqDataSetFromMatrix(assay(counts), metaData, ~Group, rowRanges = rowRanges(counts))

# Variance Stabilizing Transformation (VST)
# Normalizes data to homoscedasticity, making it suitable for PCA visualization
atacVSD <- vst(atacDDS, blind = FALSE)

# Principal Component Analysis (PCA)
# ntop = 112001 ensures we use all consensus peaks for the variance calculation
pcaData <- plotPCA(atacVSD, intgroup = "Group", ntop = 112001, returnData = TRUE) 
percentVar <- round(100 * attr(pcaData, "percentVar"))

# Visualization with ggplot2
# using ntop=112001 top features by variance
ggplot(pcaData, aes(PC1, PC2, fill=group)) +
  geom_point(size=4, shape = 21, alpha = 0.8) +
  xlab(paste0("PC1: ",percentVar[1],"% variance")) +
  ylab(paste0("PC2: ",percentVar[2],"% variance")) +
  theme_bw() + 
  theme(legend.position = "none", 
        axis.text = element_text(size = 14), 
        axis.title = element_text(size = 14))

<img src="https://www.dropbox.com/scl/fi/ovrv46y05znpf1rbjcas3/Rplot-ATAC-Dpopbox.png?rlkey=zz52h1mkjdkignie44lczwneu&st=uqn7lgha&raw=1" width="500" alt="atac_pca">

Biological replicates are tightly packed, indicating minimal technical noise and high library consistency. PC1 (49%) and PC2 (14%) capture 63% of the total variation, showing that your experimental conditions are the dominant drivers of the data.

#### Step 2: Summarizing ATAC-seq signals to motifs
Scan for TF binding sequences and calculate accessibility deviations across groups to identify TFs with the most significant global activity changes.

In [ ]:
# Load libraries
library(motifmatchr); library(TFBSTools); library(JASPAR2020); library(BSgenome.Hsapiens.UCSC.hg38)
library(chromVAR)

# Load JASPAR 2020 Motif Database
# Focuses on 'vertebrates' and the 'CORE' collection for high-confidence TF profiles
opts <- list(tax_group = "vertebrates", collection = "CORE", all_versions = FALSE)
motifsToScan <- getMatrixSet(JASPAR2020, opts)

# Correct for GC Content Bias
# Essential for ATAC-seq because Tn5 has a sequence preference and 
# different genomic regions vary in GC content, which can skew accessibility scores.
counts <- addGCBias(counts, genome = BSgenome.Hsapiens.UCSC.hg38)

# Motif Matching in Consensus Peaks
# Scans the peaks to find where the JASPAR motifs are located
motif_ix <- matchMotifs(motifsToScan, counts, genome = BSgenome.Hsapiens.UCSC.hg38)

# Compute ChromVAR Deviations
# Calculates how much the accessibility of peaks with a specific motif 
# deviates from the expected accessibility across all samples.
BiocParallel::register(BiocParallel::SerialParam()) # Run sequentially for stability
deviations <- computeDeviations(object = counts, annotations = motif_ix)
variability_Known <- computeVariability(deviations) # Rank motifs by how much they change
devZscores <- deviationScores(deviations)          # Normalized scores for visualization

# Identify the Most Variable TFs
# Merges variability metrics with Z-scores to identify 'driver' TFs
devTotal <- merge(variability_Known, devZscores, by = "row.names")
devTotal <- devTotal[order(devTotal$variability, decreasing = TRUE), ]
write.csv(devTotal, "Variability_of_JASPARmotifs_across_samples.csv", quote = FALSE)

# Display the first 10 rows and 7 columns to inspcet the most variable motifs and their associated bootstrap confidence intervals.
devTotal[1:10, 1:7]

| Row.names | Motif ID | TF            | Variability | Bootstrap Lower | Bootstrap Upper |
|-----------|----------|---------------|-------------|-----------------|-----------------|
| 463       | MA0861.1 | TP73          | 14.64       | 11.13           | 17.08           |
| 77        | MA0106.3 | TP53          | 13.45       | 10.46           | 15.43           |
| 72        | MA0101.1 | REL           | 12.86       | 9.86            | 14.83           |
| 73        | MA0102.4 | CEBPA         | 11.71       | 8.18            | 14.17           |
| 78        | MA0107.1 | RELA          | 11.32       | 8.44            | 13.15           |
| 191       | MA0525.2 | TP63          | 11.20       | 8.41            | 13.10           |
| 438       | MA0836.2 | CEBPD         | 10.54       | 7.52            | 12.50           |
| 724       | MA1636.1 | CEBPG(var.2)  | 9.80        | 7.45            | 11.43           |
| 435       | MA0833.2 | ATF4          | 9.51        | 7.66            | 10.61           |
| 172       | MA0506.1 | NRF1          | 8.90        | 6.11            | 11.03           |

The table ranks the top 10 transcription factor (TF) motifs by their variability across the samples. High variability indicates that these TFs are likely the primary drivers of the chromatin accessibility changes you observed in the PCA. The bootstrap_lower_bound and bootstrap_upper_bound are relatively narrow and do not cross zero. This confirms that the variability is statistically robust and not a result of a few outlier peaks.

In [ ]:
# Volcano Plot of TF Variability
# Motifs in the top-right corner are both highly variable and statistically significant.
ggplot(devTotal, aes(x = variability, y = -log10(p_value_adj))) +
  geom_point(alpha = 0.25, color = "red", size = 3) +
  geom_vline(xintercept = quantile(devTotal[,"variability"], .95), 
             linetype = "dashed", color = "darkblue") +
  xlab("Variability of motif-sets across samples") + 
  ylab("-log10(adjusted p.value)") + theme_classic()

<img src="https://www.dropbox.com/scl/fi/v0hzvjzc1hf1oo028x5oo/Rplot-devTotal-ATAC.png?rlkey=7319vvbqxlxliwb1f6e1hdf4g&st=eom45upn&raw=1" width="500" alt="dev_total">

The chromVAR variability plot highlights the transcription factor (TF) motifs that drive the most significant changes in chromatin accessibility across the samples. The points to the right of the dashed blue line (top 5th percentile) represent TFs with high variability scores and extremely high statistical significance ($-log10(p.adj) > 100$).


In [ ]:
# Heatmap of Top 5% Most Variable Motifs
# Visualizes how TF activity 'clusters' your samples (e.g., IL13 vs NS replicates).
devToPlot <- devTotal %>% 
  filter(variability >= quantile(devTotal[,"variability"], .95)) %>%
  select(-c(1:7)) %>% as.matrix()
rownames(devToPlot) <- devTotal$name[1:nrow(devToPlot)]

pheatmap::pheatmap(devToPlot, scale = "row", main = "Top Variable TF Motifs")

<img src="https://www.dropbox.com/scl/fi/rdlcfp9ua6aeeghxtktlj/Rplot-devToPlot-ATAC.png?rlkey=4mlgpu0j0cuhuxcvllkwcdll1&st=8sbcbfor&raw=1" width="600" alt="devtotal_heatmap">

This heatmap of chromVAR Z-scores provides a clear visualization of how specific transcription factor (TF) activity clusters the experimental groups. Each row represents a TF motif, and each column is a biological replicate.


#### Step 3: DESeq2 for differential accessibility
Identify statistically significant changes in chromatin openness between conditions to pinpoint de-novo enhancers or silenced regions.

In [ ]:
# Subset the count matrix for the comparison groups
# Replace "GroupA" and "GroupB" with your specific experimental labels
comp_groups <- c("^GroupA", "^GroupB") 
comp_counts <- assay(counts)
comp_counts <- comp_counts[, grep(paste(comp_groups, collapse = "|"), 
                                  colnames(comp_counts))]

# Refine Metadata and Set the Baseline
# 'ref' defines the control group so that positive Fold Change = More Open in Treated
comp_metaData <- metaData %>% filter(Group %in% c("GroupA", "GroupB"))
comp_metaData$Group <- factor(comp_metaData$Group, levels = c("GroupB", "GroupA"))
comp_metaData$Group <- relevel(comp_metaData$Group, ref = "GroupB")

# Run DESeq2 Differential Model
# Analyzes changes in accessibility across the defined groups
comp_atacdiff <- DESeqDataSetFromMatrix(comp_counts, comp_metaData, 
                                        design = ~ Group, 
                                        rowRanges = rowRanges(counts))
comp_atacdiff <- DESeq(comp_atacdiff)
comp_atacdiff <- results(comp_atacdiff, format = "GRanges")

# Restrict to specific regions (e.g., Promoters)
# 'toOverLap' should be your pre-defined GRanges of interest (TSS ±3kb)
comp_atacdiff <- subsetByOverlaps(comp_atacdiff, toOverLap)

# Volcano Plot Visualization
# Highlights significantly changing peaks (padj < 0.05) in black
comp_atacdiff_df <- as.data.frame(comp_atacdiff)
comp_atacdiff_df <- comp_atacdiff_df %>% 
  mutate(identity = ifelse(padj < 0.05, "Significant", "Non-Significant"))

ggplot(comp_atacdiff_df, aes(x = log2FoldChange, y = -log10(padj), color = identity)) +
  geom_point(alpha = 0.8, size = 2) +
  scale_color_manual(values = c("Significant" = "black", "Non-Significant" = "grey")) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
  theme_bw() +
  labs(title = "Differential Accessibility: GroupA vs GroupB")

<img src="https://www.dropbox.com/scl/fi/iuze7ms1poulvcl7fg5sg/Rplot-Volcano-ATAC.png?rlkey=b4sw8elsdgasp00y2tbqybm9l&st=hwcj8ahz&raw=1" width="500" alt="volcano_plot">

The volcano plot displays statistically significant shifts in chromatin accessibility, where black points represent high-confidence regulatory changes ($padj < 0.05$). Points on the right identify regions that open (potential enhancers), while those on the left identify regions that close in response to your experimental conditions.


In [ ]:
# Load libraries
library(ChIPseeker)

# Biological Annotation & Gene Mapping
# Map significant peaks to the nearest hg38 gene symbol
comp_anno <- comp_atacdiff[!is.na(comp_atacdiff$padj) & comp_atacdiff$padj < 0.05, ]
comp_anno <- annotatePeak(comp_anno, TxDb = TxDb.Hsapiens.UCSC.hg38.knownGene)
plotAnnoPie(comp_anno)

<img src="https://www.dropbox.com/scl/fi/wahg89vzbziux0g8th5pk/Rplot-annotation-ATAC.png?rlkey=i374uf184vyziq7u3nu4jdwej&st=6nh08980&raw=1" width="500" alt="annotation_plot">

This pie chart shows that your ATAC-seq peaks are overwhelmingly concentrated at promoters, with over 83% of all accessible regions located within 3kb of a Transcription Start Site (TSS). This distribution confirms high-quality enrichment for active regulatory elements and suggests that the differential accessibility results will likely have a direct and measurable impact on gene expression.

In [ ]:
# Export Results
geneID <- mapIds(x = org.Hs.eg.db, keys = comp_anno@anno$geneId, 
                 keytype = "ENTREZID", column = "SYMBOL", multiVals = "first")

mcols(comp_anno@anno)$geneID <- geneID

write.table(as.data.frame(comp_anno), "Differential_Accessibility_Results.txt", 
            quote = F, sep = "\t")

#### Step 4: Plot Footprints
Detect "dips" in signal (Tn5 protection sites) within accessible peaks to provide high-resolution evidence of physical TF occupancy.

In [ ]:
# Generate all possible 2-letter tags (AA, AB... ZZ) to scan for custom BAM tags
possibleTag <- combn(LETTERS, 2)
possibleTag <- c(paste0(possibleTag[1, ], possibleTag[2, ]),
                 paste0(possibleTag[2, ], possibleTag[1, ]))

# Setup output directory for shifted BAM files
outPath <- "split_bam"
if (dir.exists(outPath)) unlink(outPath, recursive = TRUE, force = TRUE)
dir.create(outPath)

# Loop through each BAM file to perform coordinate shifting
for(i in seq_along(bamfile)){
  bamFile <- bamfile[i]
  label <- bamfile.label[i]
  
  # Scan the first 100 reads to identify which tags are actually present in the file
  bamTop100 <- scanBam(BamFile(bamFile, yieldSize = 100), 
                       param = ScanBamParam(tag = possibleTag))[[1]]$tag
  tags <- names(bamTop100)[lengths(bamTop100) > 0]
  
  # Read the BAM file as paired-end mates (asMates = TRUE)
  gal_raw <- readBamFile(bamFile, tag = tags, asMates = TRUE, bigFile = TRUE)
  
  # Apply the ATAC-seq specific shift: +4bp for forward and -5bp for reverse strands
  # This centers the signal precisely on the Tn5 insertion site.
  shiftedBamFile <- file.path(outPath, paste0(label, "_shifted.bam"))
  gal_shifted <- shiftGAlignmentsList(gal_raw, outbam=shiftedBamFile)
}

In [ ]:
# Load libraries
library(BSgenome.Hsapiens.UCSC.hg38)
library(MotifDb)

# Query MotifDb for the Position Frequency Matrix (PFM) of the target TF
# Example uses STAT6 from the JASPAR 2022 database
target_motif <- "STAT6" 
tf_query <- query(MotifDb, c(target_motif, "jaspar2022"))
tf_pfm <- as.list(tf_query)

# Select specific shifted BAMs and their indexes for comparison (e.g., Treated vs Control)
selected_bams <- list.files(path = "split_bam", pattern = "_shifted.bam$", full.names = TRUE)[c(1,2)]
selected_indices <- list.files(path = "split_bam", pattern = "_shifted.bam.bai$", full.names = TRUE)[c(1,2)]

# Generate the footprint profile
# factorFootprints calculates the Tn5 insertion density around the motif sites
# min.score = "90%" ensures only high-confidence binding sites are analyzed
footprint_res <- factorFootprints(
    bamfiles = selected_bams, 
    index = selected_indices,
    pfm = tf_pfm[[1]], 
    genome = BSgenome.Hsapiens.UCSC.hg38,
    min.score = "90%", 
    seqlev = paste0("chr", c(1:22, "X", "Y")),
    group = c("Group1", "Group2"), # Generic group labels
    upstream = 100, 
    downstream = 100
)

<img src="https://www.dropbox.com/scl/fi/a0bkes5z25bcuviokp9ww/Rplot-plotFoorint-ATAC.png?rlkey=kwfyke7y3ci6epigc1l20aihb&st=1893ivmi&raw=1" width="600" alt="plot_footprint">

The STAT6 footprinting plot reveals a deep central "dip" in cut-site probability, providing high-resolution physical evidence that the transcription factor is actively protecting the DNA from Tn5 cleavage. The deeper footprint in the Treated group (red) compared to the Control (blue) validates increased STAT6 binding occupancy across the genome.


#### Step 5: Multi-Omics Integration
Correlate differentially accessible regions with RNA-seq data to validate whether chromatin changes directly drive gene expression.

In [ ]:
library(tidyverse)

# Clean up ATAC results for joining (example using IL13vsNS)
atac_for_join <- as.data.frame(IL13vsNSanno_atacdiff@anno) %>%
  select(geneID, log2FoldChange, padj) %>%
  rename(SYMBOL = geneID, log2FC_ATAC = log2FoldChange, padj_ATAC = padj)

# Load your external RNA-seq DE results 
# (Ensure this file has a column named 'SYMBOL')
rna_res <- read.table("RNAseq_IL13vsNS_results.txt", header = TRUE, sep = "\t")

# Join datasets
cor_data <- inner_join(atac_for_join, rna_res, by = "SYMBOL")

#### Summary of Observations
* **Transcription Factor (TF) Drivers**: TP73, TP53, and REL were identified as the most variable TF motifs across samples, indicating they are likely the primary drivers of chromatin remodeling in this system.
* **Differential Accessibility**: A non-redundant set of 111,310 peaks was utilized to pinpoint significant shifts in chromatin openness between conditions. Volcano plots identified robust "opening" (potential enhancers) and "closing" regions.
* **Physical Occupancy Validation**: STAT6 footprinting revealed a significant "dip" in cut-site probability in the Treated group compared to Control, providing high-resolution physical evidence of active TF binding rather than just sequence accessibility.
* **Global Variance**: PCA analysis showed that PC1 (49%) and PC2 (14%) together explain 63% of the total variation, confirming that experimental conditions—not technical noise—are the dominant drivers of the data.